# Build and Push Custom AutoGluon Docker Images

This notebook builds custom Docker images on top of the AWS AutoGluon DLC, allowing you to add custom dependencies or pin specific package versions.

Uses **finch** (not docker) for image builds.

In [ ]:
import boto3
import subprocess
from sagemaker import image_uris

# Configuration
REGION = boto3.Session().region_name
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]
REPO_NAME = "autogluon-custom"
IMAGE_TAG = "ag150-dlc-upgrade"
AG_VERSION = "1.5"
PY_VERSION = "py312"
INSTANCE_TYPE = "ml.m5.2xlarge"

print(f"Region: {REGION}")
print(f"Account: {ACCOUNT_ID}")
print(f"Repository: {REPO_NAME}")
print(f"Tag: {IMAGE_TAG}")

## Resolve Base Images

Get the AWS AutoGluon DLC URIs for training and inference.

In [ ]:
# Get base images from AWS DLC
training_base_image = image_uris.retrieve(
    "autogluon",
    region=REGION,
    version=AG_VERSION,
    py_version=PY_VERSION,
    image_scope="training",
    instance_type=INSTANCE_TYPE,
)

inference_base_image = image_uris.retrieve(
    "autogluon",
    region=REGION,
    version=AG_VERSION,
    py_version=PY_VERSION,
    image_scope="inference",
    instance_type=INSTANCE_TYPE,
)

print(f"Training base: {training_base_image}")
print(f"Inference base: {inference_base_image}")

# Extract DLC account ID for ECR login
dlc_account = training_base_image.split(".")[0]
print(f"DLC Account: {dlc_account}")

## ECR Authentication

Log in to both the DLC ECR (to pull base images) and your account ECR (to push custom images).

In [ ]:
# Login to DLC ECR
!aws ecr get-login-password --region {REGION} | finch login --username AWS --password-stdin {dlc_account}.dkr.ecr.{REGION}.amazonaws.com

# Login to your account ECR
!aws ecr get-login-password --region {REGION} | finch login --username AWS --password-stdin {ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com

## Build Training Image

In [ ]:
training_image = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{REPO_NAME}:{IMAGE_TAG}-training"

!cd ../docker && finch build \
    --build-arg BASE_IMAGE={training_base_image} \
    -t {training_image} \
    -f Dockerfile.training \
    .

print(f"Built training image: {training_image}")

## Build Inference Image

In [ ]:
inference_image = f"{ACCOUNT_ID}.dkr.ecr.{REGION}.amazonaws.com/{REPO_NAME}:{IMAGE_TAG}-inference"

!cd ../docker && finch build \
    --build-arg BASE_IMAGE={inference_base_image} \
    -t {inference_image} \
    -f Dockerfile.inference \
    .

print(f"Built inference image: {inference_image}")

## Smoke Test

Verify AutoGluon 1.5.0 is installed in both images.

In [ ]:
print("Testing training image:")
!finch run --rm {training_image} python -c "import autogluon; print(f'AutoGluon version: {{autogluon.__version__}}')"

print("\nTesting inference image:")
!finch run --rm {inference_image} python -c "import autogluon; print(f'AutoGluon version: {{autogluon.__version__}}')"

## Push to ECR

In [ ]:
import json

# Create ECR repository if it doesn't exist
ecr_client = boto3.client("ecr", region_name=REGION)

try:
    ecr_client.create_repository(repositoryName=REPO_NAME)
    print(f"Created repository: {REPO_NAME}")
except ecr_client.exceptions.RepositoryAlreadyExistsException:
    print(f"Repository already exists: {REPO_NAME}")

# Push training image
print(f"\nPushing training image...")
!finch push {training_image}

# Push inference image
print(f"\nPushing inference image...")
!finch push {inference_image}

## Summary

Custom images are ready to use in SageMaker training jobs and endpoints.

In [ ]:
print("Custom Image URIs:")
print(f"Training:  {training_image}")
print(f"Inference: {inference_image}")
print("\nUse these URIs in your SageMaker training jobs and endpoints.")